# Part 1 — pure-neural baseline (PPO + DQN on highway-env)

Train **PPO** (recommended) and **DQN** with discrete meta-actions, evaluate both on the same held-out seeds, **record a video of each**, and write the study metrics — all mirrored to Google Drive. Everything is driven by `configs/highway.yaml`; the `.py` modules are function-only libraries imported here.

Covers IMPLEMENTATION_PLAN.md **Part 1**.

## 1. Mount Drive + load the GitHub token (private repo)

In [ ]:
import os, warnings, logging
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=r'.*Gym has been unmaintained.*')
logging.getLogger('gym').setLevel(logging.ERROR)

from google.colab import drive
drive.mount('/content/drive')

TOKEN = None
try:
    from google.colab import userdata
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = None
if not TOKEN:
    import getpass
    TOKEN = getpass.getpass('GitHub token (input hidden): ')
assert TOKEN, 'No token provided.'
os.environ['GITHUB_TOKEN'] = TOKEN
print('Token loaded (not displayed).')

## 2. Bootstrap the repo (authenticated fetch + clone + pip install)

In [ ]:
%%bash
set -e
GH_USER=silvergjeka22
REPO=nesy-highway-driving
curl -fsSL -H "Authorization: token ${GITHUB_TOKEN}" \
  "https://raw.githubusercontent.com/${GH_USER}/${REPO}/main/bash/setup_colab.sh" \
  -o /content/setup_colab.sh
bash /content/setup_colab.sh

In [ ]:
import sys
sys.path.insert(0, '/content/nesy-highway-driving')
%cd /content/nesy-highway-driving

## 3. Load the config

In [ ]:
from utils import load_config, drive_path, save_json

cfg = load_config('configs/highway.yaml')
# Quick smoke test? Uncomment:
# from utils import deep_update
# cfg = deep_update(cfg, {'ppo': {'total_timesteps': 5000}, 'dqn': {'total_timesteps': 5000}})
cfg['paths']

## 4. Train PPO and DQN (checkpoints mirror to Drive)

In [ ]:
from agents.baselines import train_ppo, train_dqn

ppo_model = train_ppo(cfg)   # -> <drive>/checkpoints/ppo.zip
dqn_model = train_dqn(cfg)   # -> <drive>/checkpoints/dqn.zip

## 5. Evaluate both on the same held-out seeds

In [ ]:
from eval.evaluate import evaluate, record_video

ppo_metrics = evaluate(ppo_model, cfg)
dqn_metrics = evaluate(dqn_model, cfg)

save_json(ppo_metrics, drive_path(cfg, 'metrics', 'ppo.json'))
save_json(dqn_metrics, drive_path(cfg, 'metrics', 'dqn.json'))
print('PPO:', ppo_metrics['summary'])
print('DQN:', dqn_metrics['summary'])

## 6. PPO vs DQN comparison table

In [ ]:
import pandas as pd

def row(name, m):
    s = m['summary']
    return {'algo': name, 'crash_rate': round(s['crash_rate'], 3),
            'on_road_%': f"{s['on_road_pct']['mean']:.1f}\u00b1{s['on_road_pct']['std']:.1f}",
            'overtakes': f"{s['overtakes']['mean']:.2f}\u00b1{s['overtakes']['std']:.2f}",
            'return': f"{s['return']['mean']:.2f}\u00b1{s['return']['std']:.2f}",
            'length': f"{s['length']['mean']:.1f}\u00b1{s['length']['std']:.1f}"}

pd.DataFrame([row('PPO', ppo_metrics), row('DQN', dqn_metrics)])

## 7. Record a video of each model (mirror to Drive)

In [ ]:
ppo_video = record_video(ppo_model, cfg, drive_path(cfg, 'videos', 'ppo.mp4'))
dqn_video = record_video(dqn_model, cfg, drive_path(cfg, 'videos', 'dqn.mp4'))
print('PPO video dir:', ppo_video)
print('DQN video dir:', dqn_video)

## 8. Part-1 study (deliverable)
Fill in from the table + videos above:
1. **PPO vs DQN** — which wins on safety (crash rate, on-road %) vs throughput (overtakes, return)?
2. **Failure modes** — watch the videos: cutting in with too small a gap, passing on the right, tail-gating, abrupt braking. These motivate Part 2.
3. **Sensitivity** — re-run with different `shaping` weights / `env.config.vehicles_count`.

**Exit criterion:** frozen `ppo.zip` + `dqn.zip`, this metric table, two videos, all reproducible from fixed seeds. Next: `colab_2_nesy.ipynb`.